# 02 · The model

One network, two outputs from one shared set of layers:

```
image [3, 128, 128]
   │
   ▼
TRUNK: 8 conv layers (3×3 conv → BatchNorm → ReLU), two 2×2 max-pools
   │   stage 1: 128×128, 32 channels  → pool →
   │   stage 2:  64×64,  64 channels  → pool →
   │   stage 3:  32×32, 128 channels (the last two layers are "dilated": they look 2 px apart, so they see wider)
   ▼
features [128, 32, 32]   ← a 128-number description of every 4×4-pixel cell
   │                         │
   ▼                         ▼
HEATMAP HEAD                 CLASS HEAD
1×1 conv: one score          average some cells' features → linear layer → 43 scores
per cell → [1, 32, 32]       v1 "baseline": averages ALL 1,024 cells
"is the sign's center here?" v1.5 "peak":   averages only the 3×3 cells around the heatmap's peak
```

The network is **fully convolutional**: no layer cares about image size, so the same weights also run on a
512×512 photo (notebook 07).

**You'll see:** both architectures and their sizes, what an untrained model outputs and why, and how the
trained model's heatmap chooses where the class is read.

**Needs:** a trained run (`RUN_NAME`) and `data/processed/val.npz`. **Writes:** nothing.

In [ ]:
# ---- Settings: the only cell you need to edit ----
RUN_NAME = "peak_fresh_aug"  # which trained run to inspect in section 4 (any name from notebook 00's table)

In [ ]:
# Setup (the same in every notebook). All paths in this project (configs/, data/, outputs/) are relative
# to the repo root, and the scripts expect to run from there, so move this notebook's working folder up
# to the root first. Safe to run twice.
import os
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists():  # walk up until we reach the folder holding pyproject.toml
    if root == root.parent:
        raise RuntimeError("can't find the repo root: open this notebook from inside FindTheObject-CNN-Project")
    root = root.parent
os.chdir(root)
if str(root / "notebooks") not in sys.path:
    sys.path.insert(0, str(root / "notebooks"))  # lets us import notebooks/nbhelpers.py

from nbhelpers import NB_OUT, ROOT, find_run, read_json, require, run_script
print("working in", ROOT)

In [ ]:
# Stop here, with a clear message, if a file this notebook reads is missing.
require("data/processed/val.npz", how="python scripts/make_dataset.py --config configs/baseline.yaml")
RUN_DIR = find_run(RUN_NAME)  # newest outputs/checkpoints/<RUN_NAME>-<date>-<time>/
print("run:", RUN_DIR.name)

## 1. Build both architectures

`build_model(Arch(...))` makes a fresh model with random weights. The two architectures have **exactly the
same layers and weight shapes**. Only the class head's *input* differs, so they have the same parameter count,
and the same seed gives them the same starting weights.

In [ ]:
# Build an untrained baseline and peak model and count their trainable numbers ("parameters").
import torch

from findobject.models.factory import Arch, build_model

torch.manual_seed(0)
models = {arch.value: build_model(arch) for arch in Arch}
for name, m in models.items():
    print(f"{name:8}: {sum(p.numel() for p in m.parameters()):,} parameters")

In [ ]:
# Print the layer list. Each _conv block is Conv2d -> BatchNorm2d -> ReLU; "dilation=(2, 2)" marks the wide-view layers.
print(models["peak"])

In [ ]:
# Where the parameters are: almost all of them in the trunk's later layers, which have 128 input and output channels.
m = models["peak"]
for part in ("trunk", "heatmap_head", "class_head"):
    n = sum(p.numel() for p in getattr(m, part).parameters())
    print(f"{part:13} {n:9,}  ({n / sum(p.numel() for p in m.parameters()):.2%})")

## 2. Shapes in and out

The model takes a **batch** of images, `[B, 3, H, W]` with values from 0 to 1, and returns a `ModelOutput`
with two tensors:

* `heatmap_logits` `[B, 1, H/4, W/4]`: one raw score per cell. `sigmoid(score)` turns it into a probability
  between 0 and 1.
* `class_logits` `[B, 43]`: one raw score per class. `softmax` turns the 43 scores into probabilities that add
  up to 1.

"Logit" just means a raw score before that squashing step.

In [ ]:
# Push a batch of 4 random-noise images through the untrained peak model and look at the output shapes.
x = torch.rand(4, 3, 128, 128)
m = models["peak"].eval()  # eval mode: BatchNorm uses stored statistics instead of this batch's
with torch.no_grad():  # we're only looking, so don't record what's needed for training
    out = m(x)
print("heatmap_logits", tuple(out.heatmap_logits.shape))
print("class_logits  ", tuple(out.class_logits.shape))

# Fully convolutional: a bigger image just gives a bigger heatmap, with the same weights.
with torch.no_grad():
    print("512x512 input -> heatmap", tuple(m(torch.rand(1, 3, 512, 512)).heatmap_logits.shape))

## 3. What an untrained model says, and why that's on purpose

In a 32×32 heatmap, about 1 cell in 1,000 is a sign center. If the heatmap started at the default
`sigmoid(0) = 0.5` everywhere, the first lessons of training would all be "every cell is too high". The
heatmap head's bias is set so every cell **starts at 0.01** instead (`HEATMAP_PRIOR` in `models/baseline.py`).
That cuts the starting heatmap loss from about 710 to about 39.

The class head starts with no preference, so each of the 43 classes gets about 1/43 ≈ 2.3%.

In [ ]:
# Check both starting points on the random batch from above.
from findobject.models.baseline import HEATMAP_PRIOR

probs = out.heatmap_logits.sigmoid()
print(f"heatmap: mean {probs.mean():.4f}, min {probs.min():.4f}, max {probs.max():.4f}   (HEATMAP_PRIOR = {HEATMAP_PRIOR})")
cls = out.class_logits.softmax(dim=1)
print(f"class:   largest probability {cls.max():.3f}, smallest {cls.min():.3f}   (1/43 = {1 / 43:.3f})")

## 4. The trained model on one scene

`load_model` rebuilds the right architecture from the checkpoint's stored config and loads the trained weights.
Then we run one val scene through it and look inside: the heatmap, the peak cell that `peak_cells` picks, the
3×3 window whose features the class head reads, and the top 5 class probabilities.

In [ ]:
# Load the trained model onto the GPU (or CPU) and run one val scene through it.
from findobject.data.class_names import CLASS_NAMES
from findobject.data.dataset import SignScenes
from findobject.models.peak import PEAK_WINDOW, peak_cells
from findobject.training.checkpoint import load_model
from findobject.utils.device import pick_device

device = pick_device("auto")
model = load_model(RUN_DIR / "best.pt", device)
val = SignScenes(ROOT / "data/processed/val.npz")
s = val[7]
with torch.no_grad():
    out = model(s.image[None].to(device))  # [None] adds the batch dimension: [3,128,128] -> [1,3,128,128]
row, col = peak_cells(out.heatmap_logits)[0].tolist()
print(f"true class: {CLASS_NAMES[int(s.class_id)]}, true box {s.box.tolist()}")
print(f"peak cell (row {row}, col {col}) = scene pixels x {col * 4}-{col * 4 + 3}, y {row * 4}-{row * 4 + 3}")

In [ ]:
# Left: the scene with its true box. Middle: the predicted heatmap with the peak cell's 3x3 window (cyan),
# which is what the class head reads. Right: the five most likely classes.
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

heat = out.heatmap_logits.sigmoid()[0, 0].cpu()
top = out.class_logits.softmax(dim=1)[0].cpu().topk(5)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), gridspec_kw={"width_ratios": [1, 1, 1.3]})
x1, y1, x2, y2 = s.box.tolist()
axes[0].imshow(s.image.permute(1, 2, 0))
axes[0].add_patch(Rectangle((x1 - 0.5, y1 - 0.5), x2 - x1, y2 - y1, fill=False, edgecolor="lime"))
axes[0].set_title("val scene + true box"), axes[0].set_axis_off()
axes[1].imshow(heat, cmap="inferno", vmin=0, vmax=1)
half = PEAK_WINDOW // 2
axes[1].add_patch(Rectangle((col - half - 0.5, row - half - 0.5), PEAK_WINDOW, PEAK_WINDOW, fill=False,
                            edgecolor="cyan", linewidth=1.5))
axes[1].set_title(f"heatmap (peak {heat.max():.2f}) + 3x3 class window"), axes[1].set_axis_off()
bars = axes[2].barh([CLASS_NAMES[i] for i in top.indices.tolist()][::-1], top.values.tolist()[::-1])
axes[2].bar_label(bars, labels=[f"{v:.2g}" for v in top.values.tolist()[::-1]], padding=3)
axes[2].set_xscale("log")  # log scale: the winner is usually ~1.0 and the rest would be invisible bars
axes[2].set_xlim(top.values.min().item() / 10, 5), axes[2].set_title("top-5 class probabilities (log scale)")
plt.tight_layout()
plt.show()

## 5. Why reading the class at the peak helped (v1 → v1.5)

The baseline averages features over **all 1,024 cells**. A sign 16 px wide covers about 16 of those cells, so
roughly 98% of the average comes from background. The peak model reads only the 9 cells where its own heatmap
says the sign is. Note that `peak_cells` passes no gradient back into the heatmap: the heatmap learns only from
its own loss, and the class head trains whatever features sit under the window.

The checkpoints store each run's best val score, so we can compare them directly:

In [ ]:
# Val joint accuracy (peak inside the box AND class right) for v1 baseline vs v1.5 peak, read from each checkpoint.
from findobject.training.checkpoint import load_checkpoint_info

for name in ("baseline", "peak"):
    m = load_checkpoint_info(find_run(name) / "best.pt").metrics
    print(f"{name:8}: joint {m['joint_acc']:.4f}   class {m['class_acc']:.4f}   pointing {m['pointing_acc']:.4f}")

Pointing (where is the sign?) is nearly the same for both. The gain is in **class**: once the model knows
where the sign is, reading the class from just that spot is the easier job.

**Next:** `03_train.ipynb`, how these weights are learned.